#### Kube-vip Endpoint 

In [ ]:
VIP="172.16.6.85"
INTERFACE="ens192"

KVVERSION=$(curl -sL https://api.github.com/repos/kube-vip/kube-vip/releases | jq -r ".[0].name")

In [ ]:
alias kube-vip="ctr image pull ghcr.io/kube-vip/kube-vip:$KVVERSION; ctr run --rm --net-host ghcr.io/kube-vip/kube-vip:$KVVERSION vip /kube-vip"


In [ ]:
mkdir -p /etc/kubernetes/manifests/

kube-vip manifest pod \
    --interface $INTERFACE \
    --address $VIP \
    --controlplane \
    --services \
    --arp \
    --leaderElection | tee /etc/kubernetes/manifests/kube-vip.yaml

- Test

In [ ]:
ctr images ls | grep kube-vip

- Init

In [ ]:
kubeadm init --control-plane-endpoint "172.16.6.85:6443" --upload-certs --pod-network-cidr=10.244.0.0/16 --apiserver-cert-extra-sans=172.16.6.85

In [ ]:
mkdir -p $HOME/.kube
sudo cp -i /etc/kubernetes/admin.conf $HOME/.kube/config
sudo chown $(id -u):$(id -g) $HOME/.kube/config